# DistilBERT Fine-Tuning Baseline

Run this notebook from the project root or from the `baselines/` folder. It uses the shared benchmark code, writes artifacts under `artefacts/`, and evaluates `distilbert` on the fixed split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "baselines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/ilya/ML/NLP/project')

In [2]:
MODEL_NAME = "distilbert"
MAX_SAMPLES = None  # set to a small integer, e.g. 3000, for debugging
TOP_GENRES = 15
EPOCHS = 2
BATCH_SIZE = 16
TFIDF_MAX_FEATURES = 100_000

In [3]:
import pandas as pd

from baselines.config import BaselineConfig
from baselines.data_utils import prepare_data
from baselines.run_all import finalize_model_result
from baselines.transformer_baselines import train_transformer_baseline

config = BaselineConfig(
    max_samples=MAX_SAMPLES,
    top_genres=TOP_GENRES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    tfidf_max_features=TFIDF_MAX_FEATURES,
)

/home/ilya/ML/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
bundle = prepare_data(config)
bundle.stats

Found candidate tabular files:
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv (80.5 MB)
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.parquet (50.9 MB)
Choosing largest file by default: /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv
Prepared data: train=87575, val=34470, test=32986, labels=15, split=temporal_train_le_2023_val_2024_test_2025


{'source_path': '/home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv',
 'detected_columns': {'title': 'title',
  'overview': 'overview',
  'genres': 'genres',
  'release_date': 'release_date',
  'id': 'tmdb_id'},
 'rows_before_filtering': 232586,
 'rows_after_overview_filter': 193927,
 'rows_after_genre_parse': 155678,
 'rows_after_top_genre_filter': 155031,
 'selected_genre_labels': ['Drama',
  'Documentary',
  'Comedy',
  'Horror',
  'Thriller',
  'Animation',
  'Romance',
  'Music',
  'Action',
  'Crime',
  'Fantasy',
  'Science Fiction',
  'Mystery',
  'Family',
  'TV Movie'],
 'per_label_frequency': {'Drama': 54784,
  'Documentary': 43107,
  'Comedy': 29457,
  'Horror': 18098,
  'Thriller': 15386,
  'Animation': 12723,
  'Romance': 10576,
  'Music': 8382,
  'Action': 7238,
  'Crime': 6677,
  'Fantasy': 6339,
  'Science Fiction': 6141,
  'Mystery': 6245,
  'Family': 5088,
  'TV Movie': 3842},
 'train_per_label_frequency': {'Drama': 30116,
  'Documentary': 25421,
  'Comedy': 15

In [5]:
assert bundle.y_train.shape[1] == len(bundle.label_names)
assert bundle.y_val.shape[1] == len(bundle.label_names)
assert bundle.y_test.shape[1] == len(bundle.label_names)
assert {"sample_id", "text", "labels_list"}.issubset(bundle.train_df.columns)
assert bundle.train_df["text"].str.len().min() >= config.min_overview_chars

print("labels:", bundle.label_names)
print("train/val/test:", bundle.y_train.shape, bundle.y_val.shape, bundle.y_test.shape)

labels: ['Drama', 'Documentary', 'Comedy', 'Horror', 'Thriller', 'Animation', 'Romance', 'Music', 'Action', 'Crime', 'Fantasy', 'Science Fiction', 'Mystery', 'Family', 'TV Movie']
train/val/test: (87575, 15) (34470, 15) (32986, 15)


In [ ]:
result = train_transformer_baseline("distilbert", bundle, config)
metrics = finalize_model_result(result, bundle, config)
metrics 

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 18328.54it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilbert epoch 1: loss=0.1918, val_macro_f1=0.5277


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]


distilbert epoch 2: loss=0.1540, val_macro_f1=0.5377


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 15503.54it/s]


{'model': 'distilbert',
 'micro_f1': 0.6317894698738484,
 'macro_f1': 0.5319503054182433,
 'weighted_f1': 0.6338355822928463,
 'samples_f1': 0.6630117031590383,
 'precision_micro': 0.5910430071114121,
 'recall_micro': 0.6785700400450994,
 'hamming_loss': 0.08223286646860284,
 'precision_at_1': 0.7399502819377918,
 'precision_at_3': 0.4071727399502819,
 'recall_at_3': 0.8372663029722085,
 'subset_accuracy': 0.3694294549202692,
 'average_precision_micro': 0.6987258667413091}

In [7]:
prediction_path = PROJECT_ROOT / "artefacts" / "predictions" / f"{MODEL_NAME}_test_predictions.csv"
threshold_path = PROJECT_ROOT / "artefacts" / "thresholds" / f"{MODEL_NAME}_thresholds.json"
print(prediction_path)
print(threshold_path)
pd.read_csv(prediction_path).head()

/home/ilya/ML/NLP/project/artefacts/predictions/distilbert_test_predictions.csv
/home/ilya/ML/NLP/project/artefacts/thresholds/distilbert_thresholds.json


,sample_id,text,true_labels,predicted_labels,score_drama,score_documentary,score_comedy,score_horror,score_thriller,score_animation,score_romance,score_music,score_action,score_crime,score_fantasy,score_science_fiction,score_mystery,score_family,score_tv_movie
0,1052558,iPossessed [SEP] A group of celebrating friend...,Horror|Thriller,Horror,0.031536,0.001235,0.098654,0.973194,0.107678,0.015485,0.004750,0.004578,0.029602,0.004107,0.052840,0.009844,0.021950,0.001726,0.001149
1,980477,Ne Zha 2 [SEP] After a catastrophic event leav...,Animation|Action|Fantasy,Action|Fantasy,0.245586,0.001899,0.158177,0.023838,0.014801,0.130308,0.049841,0.009378,0.603283,0.004279,0.793372,0.153767,0.017536,0.032096,0.005464
2,1205229,Night of the Zoopocalypse [SEP] A wolf and mou...,Comedy|Horror|Animation|Science Fiction,Horror|Action|Science Fiction,0.029877,0.006057,0.158574,0.791610,0.137725,0.148263,0.007154,0.006902,0.613092,0.009836,0.112959,0.471077,0.018185,0.022763,0.010380
3,1084199,Companion [SEP] During a weekend getaway at a ...,Horror|Thriller|Science Fiction,Drama|Thriller|Science Fiction|Mystery,0.497115,0.005834,0.088310,0.197013,0.687688,0.012509,0.015397,0.003722,0.085147,0.058905,0.021253,0.386580,0.187832,0.005642,0.007243
4,1009640,Valiant One [SEP] With tensions between North ...,Thriller|Action,Drama|Thriller|Action,0.366726,0.011791,0.030127,0.030773,0.493570,0.014706,0.010033,0.001335,0.857522,0.046312,0.007617,0.083750,0.017542,0.004887,0.027071
